# Day 1 Smoke Test — Multi-Source RAG

In [1]:
# Make the project root importable regardless of where Antigravity's
# kernel working directory lands (same idea as running a Spring Boot
# app from the module root so package-relative imports resolve).
import sys, os
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))
os.chdir(project_root)
print("Working dir set to:", Path.cwd())

Working dir set to: d:\multi-source-rag-app


## 1. App boots + `/health` responds

In [2]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
response = client.get("/health")
print(response.status_code, response.json())
assert response.status_code == 200
print("PASS — app boots and /health responds")

200 {'status': 'ok', 'service': 'multi-source RAG', 'llm_provider': 'groq', 'embedding_model': 'BAAI/bge-small-en-v1.5'}
PASS — app boots and /health responds


c:\Users\AJAY\AppData\Local\Programs\Python\Python314\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2. Contracts instantiate and validate correctly

In [3]:
from app.contracts import (
    SourceDescriptor, RawDocument, DocumentRecord, ChunkRecord,
    IngestionJob, FailureRecord, SourceType, JobStatus, PipelineStage,
)

source = SourceDescriptor(source_type=SourceType.PDF, name="orbital-pdfs", config={"path": "data/raw/pdf"})
doc = DocumentRecord(
    source_type=SourceType.PDF, source_id=source.source_id,
    source_name="orbital_leave_policy.pdf", title="Leave Policy",
    content="Full-time employees are entitled to 24 days of paid annual leave...",
    content_hash="deadbeef123",
)
chunk = ChunkRecord(document_id=doc.document_id, chunk_index=0, content=doc.content[:50])
job = IngestionJob(source_id=source.source_id, document_id=doc.document_id)

print(source)
print(doc)
print(chunk)
print(job)
print("PASS — all contracts instantiate with expected defaults")

source_id='fdc6d8ef-49ea-4253-8b8b-a37124133eb9' source_type=<SourceType.PDF: 'pdf'> name='orbital-pdfs' config={'path': 'data/raw/pdf'} created_at=datetime.datetime(2026, 9, 25, 9, 42, 41, 73861, tzinfo=datetime.timezone.utc)
document_id='56cadad6-70e9-4e03-bae5-3002519ba91e' source_type=<SourceType.PDF: 'pdf'> source_id='fdc6d8ef-49ea-4253-8b8b-a37124133eb9' source_name='orbital_leave_policy.pdf' title='Leave Policy' content='Full-time employees are entitled to 24 days of paid annual leave...' metadata={} content_hash='deadbeef123' version=1 created_at=datetime.datetime(2026, 9, 25, 9, 42, 41, 74042, tzinfo=datetime.timezone.utc) updated_at=datetime.datetime(2026, 9, 25, 9, 42, 41, 74043, tzinfo=datetime.timezone.utc)
chunk_id='5f763527-44a7-47e7-bb58-0248667c1a70' document_id='56cadad6-70e9-4e03-bae5-3002519ba91e' chunk_index=0 content='Full-time employees are entitled to 24 days of pai' embedding=None metadata={}
job_id='bea6ac95-32d4-4a98-8ee8-28dc9fda35cf' source_id='fdc6d8ef-49e

## 3. PostgreSQL reachable

In [10]:
import psycopg
from app.config import settings

# psycopg (v3) uses a plain "postgresql://" DSN, not the SQLAlchemy
# "postgresql+psycopg://" dialect string — strip the "+psycopg" part.
dsn = settings.database_url.replace("+psycopg", "")
with psycopg.connect(dsn, connect_timeout=5) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        print(cur.fetchone()[0])
print("PASS — Postgres reachable")

PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit
PASS — Postgres reachable


## 4. Redis reachable

In [5]:
import redis
from app.config import settings

r = redis.from_url(settings.redis_url)
print("PING ->", r.ping())
print("PASS — Redis reachable")

PING -> True
PASS — Redis reachable


## 5. Qdrant reachable

In [6]:
from qdrant_client import QdrantClient
from app.config import settings

qc = QdrantClient(url=settings.qdrant_url)
print("Collections:", qc.get_collections())
print("PASS — Qdrant reachable")

Collections: collections=[]
PASS — Qdrant reachable


c:\Users\AJAY\AppData\Local\Programs\Python\Python314\Lib\site-packages\qdrant_client\qdrant_remote.py:284: UserWarning: Qdrant client version 1.19.1 is incompatible with server version 1.12.1. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


## 6. Day 1 gate summary

In [7]:
print("If every cell above printed PASS with no exceptions, the Day 1 gate is green.")
print("Next: Day 2 — source adapters (PDF, OCR fallback, CSV, SQL).")

If every cell above printed PASS with no exceptions, the Day 1 gate is green.
Next: Day 2 — source adapters (PDF, OCR fallback, CSV, SQL).
